<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Chapter 15 · Callable Bonds and Embedded Options

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/ch15_callable_bonds_embedded_options.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

A callable bond embeds an issuer-owned call option inside the bond's cash-flow rules. In this notebook you reproduce the chapter's tree valuation, compare callable and straight prices and effective durations from the same short-rate tree, and isolate the issuer's option cost. You learn to read a call schedule and its protection period, to place the exercise decision inside backward induction as the minimum of continuation value and call price, and to explain why the capped upside on falling rates produces negative convexity and a smaller effective duration than the straight bond.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Run the chapter example

The chapter script builds a recombining short-rate tree from the frozen tree inputs, values the callable and the straight bond by backward induction, and prints base prices with effective durations from a one-basis-point bump of the root rate. The callable trades below the straight bond, and its effective duration is the smaller of the two.

In [ ]:
SCRIPT = ROOT / "code" / "ch15_callable_bonds_embedded_options.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Read the printed comparison against the call mechanics.

- Confirm the tree setup: four annual steps, a root short rate of 3.5%, up and down factors of 1.15 and 0.90, and risk-neutral weights of one half.
- Check the call schedule: node values are capped at the par call price only from the year-two step onward; earlier steps use pure continuation values.
- Read the prices: the straight bond is worth 103.2088 and the callable 101.7037, so the issuer's option costs about 1.5051 per 100 of face value.
- Compare the durations: 3.7452 for the straight bond versus 2.3558 for the callable, because likely exercise caps the price gain when rates fall.
- Note the convention: effective duration is a bump-and-reprice approximation on the root rate, not an analytic derivative.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch15_*.csv"))
summary = pd.DataFrame([
    {"file": path.name,
     "rows": len(frame := pd.read_csv(path)),
     "columns": len(frame.columns),
     "missing_cells": int(frame.isna().sum().sum()),
     "fields": str(list(frame.columns))}
    for path in chapter_files
])
summary


## Chapter-specific checks

Compute both prices and effective durations from the chapter's year-two call convention, then isolate the issuer option cost and duration difference.

In [ ]:
from runpy import run_path

comparison = run_path(str(SCRIPT))["effective_duration"]()
comparison["option_cost"] = comparison.price.iloc[0] - comparison.price
comparison["duration_gap"] = comparison.duration.iloc[0] - comparison.duration
comparison.round(4)

## Mini-lab: Callable value equals straight bond minus issuer option

The chapter script drives the tree from a frozen parameter file; this mini-lab hardcodes the chapter's small four-step tree and computes both bonds directly. Backward induction starts from par at maturity, discounts expected node values at each short rate, and, once the call is active, caps node values at the par call price. The difference between the two results is the value the issuer captures and the investor gives up.

In [ ]:
rates = [[0.0350], [0.0315, 0.0402], [0.0284, 0.0362, 0.0463],
         [0.0255, 0.0326, 0.0417, 0.0532]]  # annual
coupon, notional, call_price, call_start = 4.5, 100, 100, 2
def price(is_call):  # backward induction
    values = [notional] * 5
    for step in range(3, -1, -1):
        values = [(coupon + 0.5 * (values[j] + values[j + 1]))
                  / (1 + rate) for j, rate in enumerate(rates[step])]
        if is_call and step >= call_start:
            values = [min(v, call_price) for v in values]
    return values[0]
straight, call_ = price(False), price(True)
print("straight:", round(straight, 4), " callable:", round(call_, 4))
print("issuer option:", round(straight - call_, 4))

### What to observe

The straight bond is worth 103.2101 per 100 of face value and the callable 101.7061, so the embedded call costs the investor 1.5040. The cap binds only where continuation value exceeds the par call price, which after the year-two coupon happens at the low-rate nodes. The same cap is what bends the price-yield profile toward negative convexity.

## Exercises

1. **Delay the call.** Move the first exercise step from year two to year three and recompute the option cost. *Hint:* The cap activates one step later, so fewer nodes are capped and the callable price rises toward the straight value.
2. **Raise the call price.** Set the call price to 102 and report the new callable value and option cost. *Hint:* The cap binds less often, so the callable price increases and the option cost shrinks.
3. **Trace one node.** At the year-two node with rate 0.0284, compare the continuation value of about 101.61 with the call price by hand. *Hint:* Average the two successor values, add the coupon, and discount at the node rate; the minimum with the par call price is what cuts the node value to 100.
4. **Rebuild the duration comparison.** Bump the root rate up and down by one basis point for both bonds and verify durations near 3.75 and 2.36. *Hint:* Effective duration is (P_down - P_up) / (2 * P_base * shock).
5. **Connect to the yield-to-call quote.** Explain why a yield-to-call figure quoted for the year-two call date is not a valuation of the bond. *Hint:* It assumes exercise on a single date, while the tree evaluates exercise across all future rate states after protection ends.

For more detail, see Chapter 15 of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">